# APS360 Handwriting Recognition: Step 3 (Tesseract baseline)

The proposal's baseline: **Tesseract OCR**, an off-the-shelf recognizer that was *not* trained on IAM, run on the **same IAM val and test words** as the CRNN (same `words_recovered.txt`, same filters, same writer-independent Aachen splits) and scored with the **same CER/WER code**.

| Section | What happens |
|---|---|
| Setup | Drive, code, install Tesseract, data (reuses `words_recovered.txt` from Step 1) |
| Preprocessing check | three test words before/after the white border, with Tesseract's output |
| Time estimate | times 40 words, then estimates the full run; optional subsampling (`N_TEST`) |
| Run | val + test, saved every 500 words (re-run to resume after a disconnect) |
| Report outputs | baseline metrics table, failure grid, pipeline diagram, error analysis and, if Step 2 has run, the **CRNN vs baselines** table (also includes Step 4's EMNIST baseline if it has run) |

**Runtime:** Tesseract runs on the **CPU only**, so a CPU runtime is fine (Runtime → Change runtime type → CPU); a GPU doesn't make it faster.
**Run Step 1 first** (at least once) so `words_recovered.txt` exists. Everything is saved to `outputs/report/` on Drive, next to Step 2's files.

## Mount Google Drive
Your Drive appears at `/content/drive/MyDrive`. Allow access when Colab asks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paths
`PROJECT_DIR` is the Drive folder that holds `archive.zip`. If it has been moved, the cell searches your Drive for it.
Heavy files go on Colab's local disk (`/content`), which is far faster than Drive. Results are saved back to Drive.

In [ ]:
import os, subprocess
PROJECT_DIR = "/content/drive/MyDrive/University (Mech Eng)/Year 3/Fall/APS360 (Introduction to Machine Learning)/Project/Project Progress Report"
if not os.path.exists(os.path.join(PROJECT_DIR, "archive.zip")):
    print("archive.zip not found at PROJECT_DIR; searching Drive (can take a minute)...")
    hits = [h for h in subprocess.run(["find", "/content/drive/MyDrive", "-name", "archive.zip", "-size", "+500M"],
                                      capture_output=True, text=True).stdout.split("\n") if h]
    assert hits, "Could not find archive.zip anywhere in your Drive"
    PROJECT_DIR = os.path.dirname(hits[0])

ARCHIVE    = os.path.join(PROJECT_DIR, "archive.zip")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")       # results saved back to Drive
REPORT_DIR = os.path.join(OUTPUT_DIR, "report")         # figures/tables for the progress report
OFFICIAL_WORDS_TXT = os.path.join(PROJECT_DIR, "words.txt")            # full official file, if you add it
RECOVERED_WORDS_TXT = os.path.join(PROJECT_DIR, "words_recovered.txt") # made by the label-recovery step
CODE_DIR   = "/content/aps360_htr"                      # project code (this session only)
RAW_DIR    = "/content/iam_raw"                         # archive.zip unpacked here
DATA_ROOT  = "/content/iam"                             # tidy IAM layout (symlinks into RAW_DIR)
os.environ["HTR_DATA_ROOT"] = DATA_ROOT                 # src/config.py reads this
os.makedirs(REPORT_DIR, exist_ok=True)
print("PROJECT_DIR:", PROJECT_DIR)

## Install Tesseract
`tesseract-ocr` is the OCR program (Colab's apt gives version 4.1.x, which has the LSTM engine); `pytesseract` is the Python wrapper that calls it. Takes ~30 s.

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract
!tesseract --version 2>&1 | head -1

## Write the project code
The baseline code is shown below as `%%writefile` cells. Everything else (data pipeline, metrics, report helpers) is identical to Steps 1-2 and comes from the embedded zip.

| File | What it does |
|---|---|
| `src/baselines/tesseract.py` | preprocessing (white border), Tesseract call (`--oem 1 --psm 8`), output normalization, parallel + resumable runs |
| `src/report/tesseract_report.py` | baseline metrics table, failure grid, pipeline diagram |
| `src/report/compare.py` | CRNN vs every baseline with results, on exactly the same words (matched by word ID) |
| `src/utils/metrics.py` (zip) | the same CER / WER code used for the CRNN |

### The rest of the codebase (embedded zip)
Every step notebook is generated from one repo; this one embeds code **v16** (`aps360_htr_code_v16.zip` in your project folder).
This cell unpacks the 79 files that aren't shown below (see the other notebook or the zip to read them).
The long string is just that zip, base64-encoded.

In [ ]:
import base64, io, os, shutil, sys, zipfile
# Start from a clean copy every time: delete any older code left in /content/aps360_htr,
# and forget already-imported project modules so this session can't keep using stale code.
os.chdir("/content")
shutil.rmtree("/content/aps360_htr", ignore_errors=True)
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
os.makedirs("/content/aps360_htr", exist_ok=True)
CODE_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).extractall("/content/aps360_htr")
with open("/content/aps360_htr/CODE_VERSION", "w") as f:
    f.write("v16\n")
print("unpacked", len(zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).namelist()), "files | code v16")

In [ ]:
import os
for d in sorted({os.path.dirname(f) for f in ['src/baselines/__init__.py', 'src/baselines/tesseract.py', 'src/report/tesseract_report.py', 'src/report/compare.py']}):
    os.makedirs(f'/content/aps360_htr/{d}', exist_ok=True)
%cd /content/aps360_htr

### The Tesseract baseline
Settings, the light preprocessing and the output normalization are documented at the top of the file.

In [ ]:
%%writefile /content/aps360_htr/src/baselines/__init__.py
"""Baselines the CRNN is compared against (Tesseract OCR; EMNIST is future work)."""

In [ ]:
%%writefile /content/aps360_htr/src/baselines/tesseract.py
"""Tesseract OCR baseline: an off-the-shelf recognizer run on the same IAM test words.

Settings (also in the report):
  --oem 1   use Tesseract's LSTM recognizer only (not the legacy engine)
  --psm 8   "treat the image as a single word" (IAM crops are single words)
  -l eng    English model shipped with tesseract-ocr

Preprocessing is deliberately light, so the baseline is fair but not tuned:
  grayscale + a white border of `border` px on every side. Tesseract is trained on
  text with margins; IAM crops touch the ink, and without a border it often returns nothing.
  No resizing, binarization or deskewing (Tesseract binarizes internally).

Output normalization: strip whitespace/newlines (psm 8 still sometimes splits a word) and
turn curly quotes into the straight ASCII quotes IAM uses. Nothing else is corrected.
"""
import os
import time
from concurrent.futures import ThreadPoolExecutor

from PIL import Image, ImageOps

OEM, PSM, BORDER = 1, 8, 16
_QUOTES = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2014": "-", "\u2013": "-"})


def preprocess(img: Image.Image, border: int = BORDER) -> Image.Image:
    return ImageOps.expand(img.convert("L"), border=border, fill=255)


def normalize(text: str) -> str:
    return "".join(text.split()).translate(_QUOTES)


def config(oem=OEM, psm=PSM) -> str:
    return f"--oem {oem} --psm {psm}"


def load_image(path):
    """The word image, or None if the file is missing, empty or can't be decoded."""
    try:
        with Image.open(path) as im:
            im.load()
            return im.copy() if im.size[0] and im.size[1] else None
    except Exception:
        return None


def recognize(path, border=BORDER, oem=OEM, psm=PSM, lang="eng"):
    """Tesseract's normalized output, or None if the image can't be read (the word is then skipped)."""
    import pytesseract
    img = load_image(path)
    if img is None:
        return None
    return normalize(pytesseract.image_to_string(preprocess(img, border), lang=lang, config=config(oem, psm)))


def version() -> str:
    import pytesseract
    return str(pytesseract.get_tesseract_version())


def run(items, workers=None, border=BORDER, progress=True):
    """Recognize every item (IAM WordItem). Returns (predictions, seconds).

    Each call starts one tesseract process; we run `workers` of them in parallel (one per CPU
    core), each limited to one thread (OMP_THREAD_LIMIT=1) so they don't fight over cores."""
    workers = workers or os.cpu_count() or 2
    t0, out, done = time.time(), [None] * len(items), 0
    step = max(1, len(items) // 10)
    # Only the tesseract child processes should see this: PyTorch (OpenMP) can hang with it.
    old = os.environ.get("OMP_THREAD_LIMIT")
    os.environ["OMP_THREAD_LIMIT"] = "1"
    try:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(recognize, it.path, border): i for i, it in enumerate(items)}
            for f in futures:
                out[futures[f]] = f.result()
                done += 1
                if progress and done % step == 0:
                    el = time.time() - t0
                    print(f"  {done:,}/{len(items):,} words  {el:.0f}s elapsed, ~{el / done * (len(items) - done):.0f}s left")
    finally:
        if old is None:
            os.environ.pop("OMP_THREAD_LIMIT", None)
        else:
            os.environ["OMP_THREAD_LIMIT"] = old
    return out, time.time() - t0


FIELDS = ["word_id", "writer", "ref", "hyp", "correct", "ms"]


def run_cached(items, cache_csv, workers=None, chunk=500, border=BORDER):
    """Like run(), but saves results to `cache_csv` every `chunk` words and skips words already
    there, so a Colab disconnect only loses the current chunk. Returns (rows in `items` order, seconds)
    where `ms` per row is the measured wall-clock time per word of its chunk."""
    import csv
    done = {}
    if os.path.exists(cache_csv):
        with open(cache_csv, newline="") as f:
            done = {r["word_id"]: r for r in csv.DictReader(f)}
    todo = [it for it in items if it.word_id not in done]
    if done:
        print(f"  reusing {len(items) - len(todo):,} cached predictions from {os.path.basename(cache_csv)}")
    os.makedirs(os.path.dirname(os.path.abspath(cache_csv)), exist_ok=True)
    new_file = not os.path.exists(cache_csv)
    t_all, skipped = time.time(), []
    with open(cache_csv, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new_file:
            w.writeheader()
        for start in range(0, len(todo), chunk):
            part = todo[start:start + chunk]
            hyps, secs = run(part, workers=workers, border=border, progress=False)
            ms = 1000 * secs / len(part)
            for it, h in zip(part, hyps):
                if h is None:                       # unreadable image: skipped, not cached
                    skipped.append(it.word_id)
                    continue
                row = {"word_id": it.word_id, "writer": it.writer or "", "ref": it.text, "hyp": h,
                       "correct": str(int(h == it.text)), "ms": f"{ms:.2f}"}
                done[it.word_id] = row
                w.writerow(row)
            f.flush()
            el = time.time() - t_all
            n = start + len(part)
            print(f"  {n:,}/{len(todo):,} new words, {el:.0f}s elapsed, ~{el / n * (len(todo) - n):.0f}s left")
    if skipped:
        print(f"  skipped {len(skipped)} unreadable image(s): {', '.join(skipped[:10])}")
    rows = [done[it.word_id] for it in items if it.word_id in done]
    return rows, sum(float(r["ms"]) for r in rows) / 1000

### Baseline report outputs and the CRNN vs Tesseract comparison

In [ ]:
%%writefile /content/aps360_htr/src/report/tesseract_report.py
"""Step 3: report figures, tables and numbers for the Tesseract baseline."""
import json
import random

from src.baselines import tesseract as tess
from src.report import figures, tables
from src.report.predict import original_image
from src.utils.metrics import cer, wer

pct = lambda x: f"{100 * x:.2f}"


def save_metrics(rep, preds, secs, workers, full_sizes, device="CPU"):
    """`preds`: split -> rows (word_id, ref, hyp); `secs`: split -> wall-clock seconds."""
    rows, nums = [], {"version": tess.version(), "config": tess.config(), "border_px": tess.BORDER,
                      "workers": workers, "device": device}
    for split, p in preds.items():
        refs, hyps = [r["ref"] for r in p], [r["hyp"] for r in p]
        c, w = cer(refs, hyps), wer(refs, hyps)
        ms_wall = 1000 * secs[split] / max(len(p), 1)   # wall-clock per word with `workers` in parallel
        nums[split] = {"cer": c, "wer": w, "words": len(p), "of_words": full_sizes[split],
                       "empty_outputs": sum(1 for h in hyps if not h), "ms_per_word_wall": ms_wall,
                       "ms_per_word_cpu_core": ms_wall * workers}
        rows.append({"Split": split, "Words": f"{len(p):,} of {full_sizes[split]:,}", "CER (%)": pct(c), "WER (%)": pct(w),
                     "Empty": nums[split]["empty_outputs"], "ms/word": f"{ms_wall * workers:.0f}"})
    nums["ms_per_word_wall"] = nums.get("test", nums[next(iter(preds))])["ms_per_word_wall"]
    rep.numbers("tesseract", nums)
    (rep.dir / "tesseract_metrics.json").write_text(json.dumps(nums, indent=2))
    rep.add([str(rep.dir / "tesseract_metrics.json")], "Tesseract val/test CER, WER and speed.")
    paths = tables.write_table(rows, list(rows[0]), rep.dir, "table_tesseract_metrics",
                               f"Tesseract {nums['version']} baseline (LSTM engine, {tess.config()}, "
                               f"{tess.BORDER}-px white border) on the IAM validation and test splits (random subsets if fewer words "
                               f"than the split). Empty = no text returned; ms/word = time per word on one CPU core.",
                               "tab:tesseract_metrics")
    rep.add(paths, "Baseline metrics: Tesseract CER/WER on val and test, empty outputs, speed per word per CPU core.")
    return nums


def save_failure_grid(rep, ds, preds, n=12, seed=0):
    wrong = [r for r in preds if r["ref"] != r["hyp"]]
    rows = random.Random(seed).sample(wrong, min(n, len(wrong)))
    if not rows:
        print("Tesseract got every word right: no failure grid"); return None
    by_id = {it.word_id: it for it in ds.items}
    imgs = [original_image(by_id[r["word_id"]]) for r in rows]
    paths = figures.plot_prediction_grid(imgs, [r["ref"] for r in rows], [r["hyp"] for r in rows], rep.dir,
                                         "fig_tesseract_failures")
    rep.add(paths, f"Typical Tesseract failures: {len(rows)} randomly chosen wrong test predictions "
                   "(red = predicted text; '(empty)' = Tesseract returned nothing).")
    return paths


def save_pipeline(rep):
    paths = figures.draw_tesseract_pipeline(rep.dir, border=tess.BORDER, psm=tess.PSM, oem=tess.OEM)
    rep.add(paths, "Tesseract baseline pipeline: light preprocessing (white border), the pretrained LSTM engine "
                   "in single-word mode, output normalization, and scoring with the same CER/WER code as the CRNN.")
    return paths

In [ ]:
%%writefile /content/aps360_htr/src/report/compare.py
"""CRNN vs the baselines (Tesseract, EMNIST character classifier) on exactly the same words.

Needs crnn_metrics.json + crnn_<split>_predictions.csv (Step 2) and, for each baseline that has run,
<name>_metrics.json + <name>_<split>_predictions.csv (Step 3 = tesseract, Step 4 = emnist) in the
report folder. Whichever notebook runs last builds the table with every model available.
All models are scored on the words they **all** have (matched by word ID): if a baseline was run on
a random subset of the test set, everyone is scored on that subset, so the comparison is like-for-like.
"""
import json

from src.report import errors, figures, tables
from src.report.predict import load_predictions
from src.utils.metrics import cer, wer

pct = lambda x: f"{100 * x:.2f}"

SYSTEMS = [("CRNN (ours)", "crnn"), ("Tesseract (LSTM)", "tesseract"), ("EMNIST CNN + splitter", "emnist")]
BASELINES = SYSTEMS[1:]


def _scores(rows):
    refs, hyps = [r["ref"] for r in rows], [r["hyp"] for r in rows]
    return {"cer": cer(refs, hyps), "wer": wer(refs, hyps), "words": len(rows)}


def available(d):
    """{prefix: metrics dict} for every system whose metrics + test predictions exist."""
    out = {}
    for _, p in SYSTEMS:
        if (d / f"{p}_metrics.json").exists() and (d / f"{p}_test_predictions.csv").exists():
            out[p] = json.loads((d / f"{p}_metrics.json").read_text())
    return out


def _speed(prefix, m):
    if prefix == "crnn":
        return m.get("test", {}).get("ms_per_word_batched")
    if prefix == "tesseract":
        return m.get("test", {}).get("ms_per_word_cpu_core")
    return m.get("test", {}).get("ms_per_word")


def save_comparison(rep):
    d = rep.dir
    have = available(d)
    if "crnn" not in have or len(have) < 2:
        missing = ["Step 2 (CRNN)"] if "crnn" not in have else []
        if len(have) - ("crnn" in have) < 1:
            missing.append("a baseline (Step 3 Tesseract or Step 4 EMNIST)")
        print("Comparison table not made yet; missing results from:", " and ".join(missing))
        print("It is created automatically by whichever of Steps 2-4 finishes last.")
        return None
    systems = [(n, p) for n, p in SYSTEMS if p in have]
    per_split = {}
    for split in ("val", "test"):
        preds = {p: load_predictions(d / f"{p}_{split}_predictions.csv") for _, p in systems
                 if (d / f"{p}_{split}_predictions.csv").exists()}
        if "crnn" not in preds or len(preds) < 2:
            continue
        common = set.intersection(*({r["word_id"] for r in rows} for rows in preds.values()))
        rows = {p: sorted((r for r in v if r["word_id"] in common), key=lambda r: r["word_id"]) for p, v in preds.items()}
        per_split[split] = {p: _scores(v) for p, v in rows.items()}
        per_split[split]["_rows"] = rows
    params = {"crnn": rep.numbers().get("model", {}).get("parameters"),
              "emnist": have.get("emnist", {}).get("classifier", {}).get("parameters")}
    rows, nums = [], {}
    for name, p in systems:
        row = {"Model": name}
        for split in ("val", "test"):
            s = per_split.get(split, {}).get(p)
            row[f"{split.capitalize()} CER"] = pct(s["cer"]) if s else "--"
            row[f"{split.capitalize()} WER"] = pct(s["wer"]) if s else "--"
        ms = _speed(p, have[p])
        row["ms/word"] = "" if ms is None else (f"{ms:.2f}" if ms < 10 else f"{ms:.0f}")
        row["Params"] = f"{params[p] / 1e6:.2f}M" if params.get(p) else "--"
        rows.append(row)
        nums[name] = {s: per_split[s][p] for s in per_split if p in per_split[s]}
        nums[name]["ms_per_word"] = ms
    n_test = per_split["test"]["crnn"]["words"]
    full = have["crnn"].get("test", {}).get("words")
    note = None
    if full and n_test < full:
        note = (f"All models scored on the same {n_test:,} test words (the words every model was run on); "
                f"CRNN on all {full:,} test words: CER {pct(have['crnn']['test']['cer'])}%, "
                f"WER {pct(have['crnn']['test']['wer'])}%.")
    speed = {"crnn": f"CRNN batched (128) on {have['crnn'].get('device')}",
             "tesseract": "Tesseract per word on one CPU core",
             "emnist": f"EMNIST baseline: segmentation on the CPU + batched classification on "
                       f"{have.get('emnist', {}).get('device', 'GPU')}"}
    pre = {"tesseract": "Tesseract is pretrained on printed text", "emnist": "the EMNIST CNN is trained only on EMNIST characters"}
    base_txt = "; ".join(pre[p] for _, p in systems if p in pre)
    cap = (f"CRNN vs. the baselines on the same IAM words ({n_test:,} test words from writers unseen in training). "
           f"CER and WER in %, case-sensitive, lower is better. Speed: " + "; ".join(speed[p] for _, p in systems) +
           f". Neither baseline is trained on IAM ({base_txt}).")
    paths = tables.write_table(rows, list(rows[0]), d, "table_comparison", cap, "tab:comparison", note=note)
    rep.add(paths, "Main quantitative result: CRNN vs the baselines (" + ", ".join(n for n, p in systems if p != "crnn")
            + ") CER/WER on the same val/test words, speed and size.")
    sys_rows = per_split["test"]["_rows"]
    lens = {n: errors.cer_by_length([r["ref"] for r in sys_rows[p]], [r["hyp"] for r in sys_rows[p]])
            for n, p in systems if p in sys_rows}
    p2 = figures.plot_cer_by_length(lens, d, "fig_cer_by_length_comparison",
                                    title="CER by word length: " + " vs ".join(n.split(" (")[0] for n in lens)
                                    + " (same test words)")
    rep.add(p2, "CER by word length for " + ", ".join(lens) + " on the same test words.")
    nums["note"] = note
    nums["models"] = [n for n, _ in systems]
    nums["test_words"] = n_test
    rep.numbers("comparison", nums)
    return rows

## Install missing packages
Colab already has PyTorch, OpenCV, NumPy, Pillow, matplotlib, pyarrow and pytest, so this only installs what's missing.
`editdistance` (faster CER/WER) is optional; the code falls back to pure Python without it.

In [ ]:
import importlib.util, subprocess, sys
need = {"torch": "torch", "torchvision": "torchvision", "cv2": "opencv-python", "numpy": "numpy",
        "PIL": "pillow", "matplotlib": "matplotlib", "tqdm": "tqdm", "pytest": "pytest",
        "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
        "pytesseract": "pytesseract", "editdistance": "editdistance"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
print("missing:", missing or "nothing")
for pkg in missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    note = " (optional: CER/WER use the pure-Python fallback)" if pkg == "editdistance" else ""
    print(f"  {pkg}: {'installed' if r.returncode == 0 else 'install failed' + note}")

## Unpack archive.zip (about 1.2 GB)
Copied to Colab's disk and unzipped there, which takes a few minutes. Re-running in the same session skips it.

In [ ]:
import os, shutil, time
marker = os.path.join(RAW_DIR, ".unzipped")
if os.path.exists(marker):
    print("Already unzipped this session:", RAW_DIR)
else:
    t = time.time()
    shutil.copy(ARCHIVE, "/content/archive.zip")
    print(f"copied in {time.time()-t:.0f}s"); t = time.time()
    !unzip -q -o /content/archive.zip -d {RAW_DIR}
    os.remove("/content/archive.zip")
    open(marker, "w").close()
    print(f"unzipped in {time.time()-t:.0f}s")
!ls {RAW_DIR}

## Find words.txt and the images
`setup_iam.py` finds the word images and a `words.txt` in any download layout (official tgz, Kaggle `iam_words/`, or a flat folder) and links them into `/content/iam`.

**Which labels get used** (first one that exists wins):
1. `words.txt` in your project folder: the full official file, if you download it from IAM;
2. `words_recovered.txt` in your project folder, made by the label-recovery step below;
3. the `words.txt` inside `archive.zip`. The Kaggle copy is **cut off at e07-072**, so only ~39% of IAM has labels.

In [ ]:
def pick_words_txt():
    for p in (OFFICIAL_WORDS_TXT, RECOVERED_WORDS_TXT):
        if os.path.exists(p):
            return p
    return None

def run_setup():
    wt = pick_words_txt()
    print("labels from:", wt or "the words.txt inside archive.zip")
    extra = f'--words-txt "{wt}"' if wt else ""
    !python scripts/setup_iam.py --raw {RAW_DIR} --out {DATA_ROOT} {extra}

run_setup()

## Recover the missing labels (only needed once)
**Why:** the Kaggle `words.txt` stops part-way through line `e07-072-09-`, but `archive.zip` has the images for all of IAM. Without the missing lines, about 60% of the images can't be used.

**How:** every original IAM PNG has its form's XML stored *inside the file* (a PNG text chunk). That XML gives the word's ID, its text, its tag, its bounding box and its line's segmentation status (ok/err), which are exactly the fields of a `words.txt` line. `recover_labels.py`:
1. first tries the XML inside **your own** images (fast, no download);
2. if they've been stripped, it downloads the Hugging Face copy of IAM ([priyank-m/IAM_words_text_recognition](https://huggingface.co/datasets/priyank-m/IAM_words_text_recognition), ~1.2 GB), whose PNGs still carry the XML, and matches **by word ID**. Images whose ID can't be found fall back to a pixel-hash match.

**Checks it prints, for the report:**
- *rebuilt vs original*: for words that have both, the rebuilt line must equal the original Kaggle line (expect ~100%);
- *pixel check*: the HF image is pixel-identical to yours for a sample of recovered words;
- matched / unmatched / ambiguous counts. Below 90%, nothing is written and training uses the labelled subset.

The result is saved to your project folder as `words_recovered.txt` (original lines first, then a `#--- recovered entries below ---` section), so later sessions and the Step 2 notebook reuse it. Recovered words keep their real IAM IDs, so the **Aachen writer-disjoint splits still apply**.

In [ ]:
if os.path.exists(OFFICIAL_WORDS_TXT):
    print("Using the official full words.txt; no recovery needed.")
elif os.path.exists(RECOVERED_WORDS_TXT):
    print("Reusing", RECOVERED_WORDS_TXT)
else:
    !python scripts/recover_labels.py --data-root {DATA_ROOT} --out "{RECOVERED_WORDS_TXT}" \
        --hf-cache /content/hf_iam --report "{OUTPUT_DIR}/label_recovery.json" \
        --preview "{OUTPUT_DIR}/label_recovery_preview.png"
    if os.path.exists(RECOVERED_WORDS_TXT):
        run_setup()          # re-link with the recovered labels
        from IPython.display import Image as _I, display
        display(_I(os.path.join(OUTPUT_DIR, "label_recovery_preview.png")))

## Run the tests
The baseline (preprocessing, normalization, a real Tesseract call on a printed word, resuming), the report helpers and CER/WER. Every line should say `passed`.

In [ ]:
!python -m pytest -q tests/test_tesseract.py tests/test_report.py tests/test_metrics.py

## Load the same splits as the CRNN
Same `make_datasets()` as Steps 1-2: same labels, filters and Aachen writer-independent splits. No preloading: Tesseract reads each original PNG itself.

In [ ]:
from src.training.data import load_data
datasets, charset, stats = load_data(DATA_ROOT, splits=(), preload=False, check_images=False)
print({s: len(datasets[s]) for s in ("val", "test")})

## Preprocessing check
Tesseract gets the **original-resolution** word image (not the 32-px version the CRNN uses), with light preprocessing:
- grayscale, plus a **16-px white border** on every side. Tesseract is trained on text with margins; IAM crops touch the ink, and without a border it often returns nothing;
- no resizing, binarization or deskewing: Tesseract binarizes internally, and heavier tuning would no longer be an off-the-shelf baseline.

Settings: `--oem 1` (LSTM engine only), `--psm 8` (treat the image as one word), `-l eng`.
The output is normalized by removing whitespace/newlines and turning curly quotes into ASCII quotes; nothing else is corrected.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from PIL import Image as PILImage
from src.baselines import tesseract as tess
print("Tesseract", tess.version(), "|", tess.config())
items = [it for it in datasets["test"].items[:20] if tess.load_image(it.path) is not None][:3]
fig, axes = plt.subplots(len(items), 2, figsize=(8, 1.4 * len(items)))
for (a, b), it in zip(axes, items):
    img = tess.load_image(it.path).convert("L")
    a.imshow(img, cmap="gray"); a.set_title(f"original ({img.width}x{img.height})", fontsize=8)
    b.imshow(tess.preprocess(img), cmap="gray"); b.set_title(f"with border -> Tesseract: {tess.recognize(it.path)!r}  (true: {it.text!r})", fontsize=8)
    for ax in (a, b): ax.axis("off")
plt.tight_layout(); plt.show()

## Settings and time estimate
`N_VAL` / `N_TEST` = number of words to run (`None` = the whole split). For a fair comparison, use the full test split if you have the time; otherwise a random subset (e.g. `N_TEST = 2000`) is fine, because the CRNN is then scored on **exactly the same** words in the comparison table.

The cell times 40 words and prints how long each choice would take. Colab's CPU runtime has 2 cores, and each word takes about 0.05-0.1 s per core, so the full test split (~20,000 words) is roughly 10-25 minutes.

In [ ]:
import os, random, time
N_VAL  = None    # e.g. 2000 for a quicker run; None = all validation words
N_TEST = None    # e.g. 2000; None = all test words
WORKERS = os.cpu_count() or 2

probe = random.Random(1).sample(datasets["test"].items, min(40, len(datasets["test"])))
_, secs = tess.run(probe, workers=WORKERS, progress=False)
ms = 1000 * secs / len(probe)
print(f"{ms:.0f} ms/word with {WORKERS} parallel workers ({ms * WORKERS:.0f} ms per word per core)")
for split in ("val", "test"):
    n_all = len(datasets[split])
    print(f"  {split}: all {n_all:,} words ~{n_all * ms / 60000:.0f} min | 2,000 words ~{2000 * ms / 60000:.1f} min")

def pick(split, n):
    items = datasets[split].items
    return items if not n or n >= len(items) else random.Random(0).sample(items, n)

## Run Tesseract on val and test
Results are appended to `outputs/report/tesseract_<split>_predictions.csv` every 500 words. **If Colab disconnects, re-run the notebook**: words already done are skipped. To start over, delete those two CSV files.

In [ ]:
preds, secs, full_sizes = {}, {}, {}
for split, n in (("val", N_VAL), ("test", N_TEST)):
    items = pick(split, n)
    full_sizes[split] = len(datasets[split])
    print(f"{split}: {len(items):,} of {full_sizes[split]:,} words")
    preds[split], secs[split] = tess.run_cached(items, os.path.join(REPORT_DIR, f"tesseract_{split}_predictions.csv"),
                                                workers=WORKERS, chunk=500)
    refs, hyps = [r["ref"] for r in preds[split]], [r["hyp"] for r in preds[split]]
    from src.utils.metrics import cer, wer
    print(f"  {split}: CER {cer(refs, hyps):.4f} | WER {wer(refs, hyps):.4f} | empty outputs {sum(not h for h in hyps)}")

# Report outputs
Saved to `outputs/report/` (same folder as Step 2): figures as PNG (300 dpi) + PDF, tables as CSV + LaTeX booktabs `.tex`.

In [ ]:
from IPython.display import Image, Markdown, display
from src.report.manifest import Report
from src.report import tesseract_report as tr, crnn_report as cr, compare
from src.report.tables import csv_to_markdown
rep = Report(REPORT_DIR, step=3)
def show(paths):
    for p in paths or []:
        if p.endswith(".png"):
            display(Image(p, width=700))
show_table = lambda name: display(Markdown(csv_to_markdown(os.path.join(REPORT_DIR, name + ".csv"))))

### Baseline metrics
`table_tesseract_metrics`: CER, WER, how often Tesseract returned nothing, and time per word per CPU core.

In [ ]:
tnums = tr.save_metrics(rep, preds, secs, WORKERS, full_sizes, device="CPU")
show_table("table_tesseract_metrics")

### Qualitative: typical failures
`fig_tesseract_failures`: 12 randomly chosen **wrong** Tesseract predictions (red), at original resolution.

In [ ]:
show(tr.save_failure_grid(rep, datasets["test"], preds["test"], n=12, seed=0))

### Error analysis
Same analysis as for the CRNN: top character substitutions, most common wrong words, CER by word length.

In [ ]:
_, tbd, tpaths = cr.save_error_analysis(rep, preds["test"], system="Tesseract", prefix="tesseract")
print("test errors:", tbd)
show(tpaths[:2]); show_table("table_tesseract_word_errors")
show(cr.save_cer_by_length(rep, preds["test"], system="Tesseract", prefix="tesseract")[:2])

### Baseline pipeline diagram
`fig_tesseract_pipeline`.

In [ ]:
show(tr.save_pipeline(rep))

### CRNN vs the baselines
`table_comparison` (+ `fig_cer_by_length_comparison`): the CRNN, Tesseract and (if Step 4 has run) the EMNIST baseline on **the same words** (matched by word ID). Needs Step 2's `crnn_metrics.json` and `crnn_*_predictions.csv` in `outputs/report/`. If Step 2 hasn't run yet, this says so, and the last notebook to finish makes the table.

In [ ]:
if compare.save_comparison(rep):
    show_table("table_comparison")
    show([os.path.join(REPORT_DIR, "fig_cer_by_length_comparison.png")])

### Summary and list of files

In [ ]:
from src.report.manifest import future_work
summary = rep.write_summary(future_work(rep))   # the EMNIST item drops out once Step 4 has results
display(Markdown(open(summary).read()))

In [ ]:
print("Saved report files:")
rep.list_files();

## Notes for the report
- Tesseract is a **pretrained printed-text OCR** with no IAM training, so a high CER on cursive handwriting is expected; that gap is the point of the baseline.
- Report the settings (`--oem 1 --psm 8`, 16-px border, the normalization) so the baseline is reproducible.
- The second baseline (EMNIST character classifier) is **Step 4**; run it to add it to the comparison.